# Rust ReCom

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

The Rust ReCom runner uses [rustrecom](https://github.com/mggg/rustrecom) to generate
conventional ReCom chains at native-code speed. It also exposes region-aware proposals, a
reversible variant, and two score-directed optimization workflows. The Python objects validate
and serialize the run;
`rustrecom` performs the sampling inside Docker.

## Input and paths

`RecomRunnerConfig` takes a NetworkX node-link JSON dual graph. Each node needs the population
and initial-assignment attributes specified by the run. Region-aware, weighted, constrained, and
optimized runs need the additional node or edge attributes used by those settings.

The input directory is mounted read-only. Results go into `output/<input stem>/`, and logs go
into `logs/<input stem>/`.

In [ ]:
from gerrytools.mgrp import RecomRunInfo, RecomRunnerConfig

config = RecomRunnerConfig(
    json_file_path="data/dual_graph.json",
    output_folder="output",
    log_folder="logs",
)

## Configure an ordinary chain

`pop_col` and `assignment_col` connect the sampler to graph attributes. `pop_tol=0.01` allows
district populations within one percent of ideal, and `rng_seed` makes the native sampler's
random stream reproducible for the same input and configuration. Variant `B` chooses an
adjacent district pair before building a minimum spanning tree.

In [ ]:
run = RecomRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    variant="B",
    n_steps=10_000,
    pop_tol=0.01,
    writer="bendl",
    rng_seed=2026,
)

Once Docker is running and the graph exists at the configured path, execute the run with the
shared container context manager:

```python
from gerrytools.mgrp import RunContainer

with RunContainer(config) as container:
    output_path = container.run(run)
```

`output_path` is the primary host file. Stderr, including engine diagnostics, is written to
the run's log rather than mixed into the assignment stream. The temporary container is removed
when the `with` block exits, even if the run raises an exception.

## Proposal variants

A variant combines a rule for selecting adjacent districts with a spanning-tree distribution:

| Variant | Pair selection | Spanning tree | Additional requirement |
| --- | --- | --- | --- |
| `A` | Select a cut edge | Minimum spanning tree | None |
| `B` | Select an adjacent district pair | Minimum spanning tree | None |
| `C` | Select a cut edge | Uniform spanning tree | None |
| `D` | Select an adjacent district pair | Uniform spanning tree | None |
| `R` | Reversible ReCom | Reversible kernel | Positive `balance_ub` |
| `AW` | Select a cut edge | Region-aware minimum spanning tree | `region_weights` |
| `BW` | Select an adjacent district pair | Region-aware minimum spanning tree | `region_weights` |

The long `rustrecom` spellings, such as `district-pairs-mst`, are also accepted and normalized.
The choice between cut edges and district pairs changes how adjacent pairs are weighted; the
choice between minimum and uniform spanning trees changes the tree distribution used for the
balanced cut.

### Region-aware proposals

Region-aware variants add the configured surcharge to graph edges whose endpoints have
different values in a region column. Larger positive surcharges make those edges less likely
to appear in the minimum spanning tree, encouraging the proposal to preserve the region.
Multiple columns can be weighted at once.

In [ ]:
region_aware = RecomRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    variant="BW",
    region_weights={"COUNTY": 2.0, "MUNICIPALITY": 0.5},
    n_steps=10_000,
    writer="bendl",
    rng_seed=2026,
)

For variants `A`, `B`, `AW`, and `BW`, `edge_weight_keys` can add numeric edge attributes to
the spanning-tree weights. This is separate from `region_weights`: the former reads values
already stored on each edge, while the latter derives a crossing surcharge from node labels.

### Population and parallelism

| Setting | Meaning |
| --- | --- |
| `pop_tol` | Permitted relative deviation from ideal population |
| `target_pop` | Explicit ideal population; omit it to derive total population divided by district count |
| `n_threads` | Number of native proposal-generation threads |
| `batch_size` | Proposals assigned to one batch of threaded work |
| `rng_seed` | Seed for the native random-number generator |

The defaults use one thread and a batch size of one. Both settings are part of the effective
configuration and can affect reproducibility.

## Choose an output writer

| Writer | Use |
| --- | --- |
| `bendl` | One self-describing file containing the graph, metadata, and compact assignment stream |
| `ben` | Compact assignment stream when the graph and metadata are managed separately |
| `canonical` | Standard JSONL records with `assignment` and `sample` fields |
| `jsonl` / `jsonl-full` | Native JSONL output; `sum_cols` requests district aggregates |
| `tsv` | Tabular native output |
| `pcompress` | PCompress assignment stream |
| `assignments` / `canonicalized-assignments` | One assignment file per recorded step |

BENDL is usually the most convenient archive because it retains the graph and run metadata in
the same file as the plans. `bendl_graph_order` can be `none`, `rcm`, `mlc`, or `key:<attr>`;
reordering can improve compression and is recorded so the original node order remains
recoverable. Non-BENDL file outputs receive a separate metadata sidecar.

In [ ]:
portable_run = RecomRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    variant="B",
    n_steps=100_000,
    writer="bendl",
    bendl_graph_order="mlc",
    rng_seed=2026,
)

## Add a hard district constraint

Rust ReCom currently accepts one constraint per run. `district_share_floor` rejects a proposal
when either changed district falls below the requested numerator-to-denominator share. The
initial assignment must satisfy the same floor.

In [ ]:
from gerrytools.mgrp import Constraints

share_floor = Constraints().district_share_floor(
    numerator_col="BVAP",
    denominator_cols=["VAP"],
    threshold=0.40,
)

constrained_run = RecomRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    variant="B",
    constraint=share_floor,
    n_steps=10_000,
)

## Apply GerryChain updaters while streaming

For metrics that are already expressed as GerryChain updaters,
`mcmc_run_with_updaters()` forces canonical assignments to stdout, reconstructs each plan as a
`Partition`, and yields the updater values. It works well for a modest number of Python-only
metrics; native output followed by the scoring module is generally a better fit for a
large ensemble.

The graph's node labels must be the integers `0` through `n - 1` in ascending order because the
native assignment vectors are positional.

In [ ]:
def district_count(partition):
    return len(partition.parts)


streaming_run = RecomRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    variant="B",
    n_steps=100,
    updaters={"district_count": district_count},
)

```python
from gerrytools.mgrp import RunContainer

with RunContainer(config) as container:
    for result, error in container.mcmc_run_with_updaters(streaming_run):
        if error is not None:
            print(error, end="")
        elif result is not None:
            print(result)
```

The engine's exit status is checked when the iterator is exhausted.

## Optimize an objective

Rust ReCom has two search workflows in addition to ordinary sampling. Short bursts accept
valid proposals within a fixed-length burst, then restart from the best plan seen in that
burst. A tilted run keeps one continuous chain, always accepts improvements, and sometimes
accepts worse proposals. These are optimization heuristics, not substitutes for a chain drawn
from the ordinary ReCom distribution.

Both workflows take one `Objective` specification and write a score CSV beside the recorded
plans. `maximize` controls direction: count-like objectives are normally maximized, while
deviation objectives are normally minimized.

In [ ]:
from gerrytools.mgrp import Objective, ShortBurstsRunInfo, TiltedRunInfo

objective = Objective.gingles_partial(
    threshold=0.50,
    min_pop="BVAP",
    total_pop="VAP",
)

short_bursts = ShortBurstsRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    objective=objective,
    burst_length=25,
    n_steps=5_000,
    maximize=True,
    variant="B",
)

tilted = TiltedRunInfo(
    pop_col="TOTPOP",
    assignment_col="DISTRICT",
    objective=objective,
    n_steps=5_000,
    maximize=True,
    variant="B",
    accept_rule="exponential",
    acceptance_beta=2.0,
)

Tilted acceptance rules differ in how they tolerate a worse score:

| Rule | Control | Behavior |
| --- | --- | --- |
| `fixed` | `accept_worse_prob` | Constant probability, independent of the loss |
| `linear` | `acceptance_beta` | Probability falls linearly with score loss |
| `exponential` | `acceptance_beta` | Metropolis-style exponential penalty for score loss |

Available objective builders cover target-share deviations, threshold and banded Gingles
scores, election wins across one or more elections, and Polsby-Popper compactness. The
[MGRP API](../../api/mgrp.rst) documents each formula and required graph column.

## Inspect the effective configuration

`run_config()` exposes the exact validated document sent to `rustrecom`. Inspect it before a
run or store it with the project's other provenance. The runner also provides `output_file()`,
`expected_files()`, and `log_file()` for the resolved host paths.

In [ ]:
config.run_config(portable_run)

## Related

- [Ensemble runners overview](../mgrp.md)
- [Recording chains with BEN](../ben.ipynb)
- [Scoring BENDL files](../scoring/bendl.ipynb)
- [MGRP API](../../api/mgrp.rst)